# NYC Airbnb SQL Analytics Pipeline
## Notebook 1: Data Exploration & Pipeline Verification

**Author:** Mike Ichikawa  
**Date:** December 2025

First pass at the raw data — understand the shape, distributions, nulls, and quirks before building the analytical layer.

This notebook documents: what the raw data looks like, what cleaning decisions were made, and what the staging → mart transformation does.

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sqlalchemy import create_engine, text

import config

engine = create_engine(config.DATABASE_URL)
pd.set_option('display.max_columns', 30)
pd.set_option('display.float_format', '{:.2f}'.format)
print('Ready. DB:', config.DATABASE_URL)

In [ ]:
# Load raw staging table
with engine.connect() as conn:
    raw = pd.read_sql(text('SELECT * FROM stg_listings_raw LIMIT 5000'), conn)

print(f'Shape: {raw.shape}')
print(f'Columns: {list(raw.columns)}')
raw.head(3)

In [ ]:
# Null audit
null_pct = (raw.isnull().sum() / len(raw) * 100).sort_values(ascending=False)
null_pct = null_pct[null_pct > 0]

fig, ax = plt.subplots(figsize=(10, 6))
null_pct.plot(kind='barh', ax=ax, color='#D7191C', alpha=0.8)
ax.set_xlabel('% Null')
ax.set_title('Null Percentage by Column (staging layer)')
ax.axvline(5, color='orange', linestyle='--', label='5% threshold')
ax.legend()
plt.tight_layout()
plt.savefig('../data/outputs/nb_exploration_nulls.png', dpi=120, bbox_inches='tight')
plt.show()

print('High-null columns (>20%):',
      null_pct[null_pct > 20].index.tolist())

In [ ]:
# Price distribution — raw vs filtered
with engine.connect() as conn:
    prices_raw  = pd.read_sql(text('SELECT price FROM stg_listings_raw WHERE price IS NOT NULL'), conn)['price']
    prices_clean = pd.read_sql(text('SELECT price FROM stg_listings_clean'), conn)['price']

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Price Distribution: Raw vs. Filtered')

axes[0].hist(prices_raw.clip(0, 2000), bins=80, color='#D7191C', alpha=0.7, edgecolor='white')
axes[0].set_title(f'Raw (n={len(prices_raw):,}, including outliers)')
axes[0].set_xlabel('Price ($)')
axes[0].axvline(prices_raw.median(), color='black', linestyle='--',
                label=f'Median: ${prices_raw.median():.0f}')
axes[0].legend()

axes[1].hist(prices_clean.clip(0, 600), bins=80, color='#2C7BB6', alpha=0.7, edgecolor='white')
axes[1].set_title(f'Cleaned, capped $10-2000 (n={len(prices_clean):,})')
axes[1].set_xlabel('Price ($, shown up to $600)')
axes[1].axvline(prices_clean.median(), color='black', linestyle='--',
                label=f'Median: ${prices_clean.median():.0f}')
axes[1].legend()

plt.tight_layout()
plt.savefig('../data/outputs/nb_exploration_price_dist.png', dpi=120, bbox_inches='tight')
plt.show()

print(f'Rows removed by price filter: {len(prices_raw) - len(prices_clean):,} ({100*(len(prices_raw)-len(prices_clean))/len(prices_raw):.1f}%)')

In [ ]:
# Verify mart tables exist and have expected counts
tables = ['stg_listings_raw', 'stg_listings_clean', 'mart_listings',
          'mart_pricing_by_neighborhood', 'mart_host_stats']

print('Pipeline table row counts:')
print('-' * 40)
with engine.connect() as conn:
    for t in tables:
        try:
            n = conn.execute(text(f'SELECT COUNT(*) FROM {t}')).fetchone()[0]
            print(f'  {t:<40} {n:>8,}')
        except Exception as e:
            print(f'  {t:<40}  NOT FOUND ({e})')
print('-' * 40)
print('All pipeline stages verified ✓')